In [1]:
import warnings
warnings.filterwarnings('ignore')
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(temperature=0.9, model='gpt-3.5-turbo')

prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Replace LLMChain with LCEL
chain = prompt | llm

product = "Queen Size Sheet Set"

response = chain.invoke({"product": product})

print(response.content)

"Royal Bedding"


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(temperature=0.9, model='gpt-3.5-turbo')

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = first_prompt | llm | StrOutputParser()


# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)

# Chain 2
chain_two = second_prompt | llm | StrOutputParser()


# Input
product = "Queen Size Sheet Set"

# ---- Equivalent of SimpleSequentialChain ----

# Step 1
output1 = chain_one.invoke({"product": product})
print("Chain 1 Output:", output1)

# Step 2 (feeds into next chain)
output2 = chain_two.invoke({"company_name": output1})
print("Chain 2 Output:", output2)

# Final result (same as .run())
print("\nFinal Output:", output2)

Chain 1 Output: Royal Comfort Bedding
Chain 2 Output: Royal Comfort Bedding provides luxurious and comfortable bedding products including sheets, pillows, and duvets for a restful night's sleep.

Final Output: Royal Comfort Bedding provides luxurious and comfortable bedding products including sheets, pillows, and duvets for a restful night's sleep.


In [5]:
import pandas as pd
df = pd.read_csv('Data.csv')
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top init...
3,Pillows Insert,This is the best throw pillow filler on Amazon...
4,Milk Frother Handheld,"I loved this product, but they only seem to la..."


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(temperature=0.9, model='gpt-3.5-turbo')

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)

chain_one = first_prompt | llm | StrOutputParser()


# prompt template 2: summarize
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)

chain_two = second_prompt | llm | StrOutputParser()


# prompt template 3: detect language
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)

chain_three = third_prompt | llm | StrOutputParser()


# prompt template 4: follow-up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)

chain_four = fourth_prompt | llm | StrOutputParser()


# ---- Sequential Execution (Equivalent to SequentialChain) ----

review = df.Review[5]

# Step 1
English_Review = chain_one.invoke({"Review": review})
print("English_Review:", English_Review)

# Step 2
summary = chain_two.invoke({"English_Review": English_Review})
print("Summary:", summary)

# Step 3
language = chain_three.invoke({"Review": review})
print("Language:", language)

# Step 4
followup_message = chain_four.invoke({
    "summary": summary,
    "language": language
})
print("Follow-up message:", followup_message)


# Final output (same as SequentialChain output_variables)
result = {
    "English_Review": English_Review,
    "summary": summary,
    "followup_message": followup_message
}

print("\nFinal Result:", result)

English_Review: La calidad del sonido es increíble para el tamaño. Sin embargo, la duración de la batería podría ser mejor.
Summary: El altavoz tiene un sonido increíble para su tamaño, aunque la duración de la batería podría mejorar.
Language: English
Follow-up message: Thank you for your feedback on the speaker. I'm glad to hear that you think it has amazing sound quality for its size. We will definitely take your suggestion about improving the battery life into consideration for future models. Thank you for bringing this to our attention.

Final Result: {'English_Review': 'La calidad del sonido es increíble para el tamaño. Sin embargo, la duración de la batería podría ser mejor.', 'summary': 'El altavoz tiene un sonido increíble para su tamaño, aunque la duración de la batería podría mejorar.', 'followup_message': "Thank you for your feedback on the speaker. I'm glad to hear that you think it has amazing sound quality for its size. We will definitely take your suggestion about impro

In [10]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [11]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [13]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

llm = ChatOpenAI(temperature=0, model='gpt-3.5-turbo')
destination_chains = {}

for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]

    prompt = ChatPromptTemplate.from_template(prompt_template)
    chain = prompt | llm | StrOutputParser()

    destination_chains[name] = chain

default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = default_prompt | llm | StrOutputParser()

from langchain_core.prompts import PromptTemplate

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
)

router_chain = router_prompt | llm | StrOutputParser()

def route(input_text):
    # Step 1: Ask router LLM
    router_output = router_chain.invoke({"input": input_text})
    
    print("\nRouter raw output:\n", router_output)

    # Step 2: Extract JSON from markdown
    try:
        json_str = router_output.split("```json")[1].split("```")[0]
        parsed = json.loads(json_str)
    except Exception as e:
        print("Parsing failed, using default chain.")
        return default_chain.invoke({"input": input_text})

    destination = parsed.get("destination", "DEFAULT")
    next_inputs = parsed.get("next_inputs", input_text)

    print("Destination:", destination)
    print("Next Inputs:", next_inputs)

    # Step 3: Route to correct chain
    if destination in destination_chains:
        return destination_chains[destination].invoke({"input": next_inputs})
    else:
        return default_chain.invoke({"input": next_inputs})